In [ ]:


# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)

# Make use of a GPU or MPS (Apple) if one is available.
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from scipy.stats import zscore
import pandas as pd
import logging
import os
logging.disable(logging.WARNING)

# Read the data set
df = pd.read_csv("/content/jh-simple-dataset.csv", na_values=['NA', '?'])

# Generate dummies for job
df = pd.concat([df, pd.get_dummies(df['job'], prefix="job", dtype=int)], axis=1)
df.drop('job', axis=1, inplace=True)

# Generate dummies for area
df = pd.concat([df, pd.get_dummies(df['area'], prefix="area", dtype=int)], axis=1)
df.drop('area', axis=1, inplace=True)

# Missing values for income
med = df['income'].median()
df['income'] = df['income'].fillna(med)

# Standardize ranges
df['income'] = zscore(df['income'])
df['aspect'] = zscore(df['aspect'])
df['save_rate'] = zscore(df['save_rate'])
df['age'] = zscore(df['age'])
df['subscriptions'] = zscore(df['subscriptions'])

# Convert to numpy - Classification
x_columns = df.columns.drop('product').drop('id')
x = df[x_columns].values
dummies = pd.get_dummies(df['product'])
products = dummies.columns
y = dummies.values

import pandas as pd
import numpy as np
import time
import statistics
from scipy.stats import zscore
from sklearn import metrics
from sklearn.model_selection import StratifiedShuffleSplit

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert data to PyTorch tensors
x_tensor = torch.FloatTensor(x).to(device)
y_tensor = torch.LongTensor(np.argmax(y, axis=1)).to(device)

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, dropout, neuronPct, neuronShrink):
        super(NeuralNetwork, self).__init__()

        layers = []
        neuronCount = int(neuronPct * 5000)
        layer = 0

        prev_count = input_dim
        while neuronCount > 25 and layer < 10:
            layers.append(nn.Linear(prev_count, neuronCount))
            prev_count = neuronCount
            layers.append(nn.PReLU())
            layers.append(nn.Dropout(dropout))
            neuronCount = int(neuronCount * neuronShrink)
            layer += 1

        layers.append(nn.Linear(prev_count, y.shape[1]))
        layers.append(nn.Softmax(dim=1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# Create and print the model
model = NeuralNetwork(x.shape[1], 0.2, 0.1, 0.25).to(device)
print(model)

# Evaluation function
SPLITS = 2
EPOCHS = 500
PATIENCE = 10

def evaluate_network(learning_rate=1e-3, dropout=0.2, neuronPct=0.1, neuronShrink=0.25):
    boot = StratifiedShuffleSplit(n_splits=SPLITS, test_size=0.1)
    mean_benchmark = []
    epochs_needed = []

    for train, test in boot.split(x, np.argmax(y, axis=1)):
        x_train = x_tensor[train]
        y_train = y_tensor[train]
        x_test = x_tensor[test]
        y_test = y_tensor[test]

        model = NeuralNetwork(x.shape[1], dropout=dropout, neuronPct=neuronPct, neuronShrink=neuronShrink).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        dataset_train = TensorDataset(x_train, y_train)
        loader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(EPOCHS):
            model.train()
            for batch_x, batch_y in loader_train:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                outputs_test = model(x_test)
                val_loss = criterion(outputs_test, y_test).item()

            if val_loss < best_loss:
                best_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                epochs_needed.append(epoch)
                break

        # Evaluate
        with torch.no_grad():
            model.eval()
            pred = model(x_test).cpu().numpy()
            y_compare = y_test.cpu().numpy()
            score = metrics.log_loss(y_compare, pred)
            mean_benchmark.append(score)

    return -statistics.mean(mean_benchmark)

print(evaluate_network(learning_rate=1e-3, dropout=0.2, neuronPct=0.1, neuronShrink=0.25))
print(evaluate_network(dropout=0.2, neuronPct=0.1, neuronShrink=0.25))

# HIDE OUTPUT
!pip install bayesian-optimization

from bayes_opt import BayesianOptimization
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Bounded region of parameter space
pbounds = {
    'dropout': (0.0, 0.499),
    'learning_rate': (0.0, 0.1),
    'neuronPct': (0.01, 1),
    'neuronShrink': (0.01, 1)
}

optimizer = BayesianOptimization(
    f=evaluate_network,
    pbounds=pbounds,
    verbose=2,
    random_state=1,
)

start_time = time.time()
optimizer.maximize(init_points=10, n_iter=20)
time_took = time.time() - start_time

print(f"Total runtime: {hms_string(time_took)}")
print(optimizer.max)

Using device: cpu
NeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=47, out_features=500, bias=True)
    (1): PReLU(num_parameters=1)
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=500, out_features=125, bias=True)
    (4): PReLU(num_parameters=1)
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=125, out_features=31, bias=True)
    (7): PReLU(num_parameters=1)
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=31, out_features=7, bias=True)
    (10): Softmax(dim=1)
  )
)
-4.020523602576961
-3.4643732868759747
|   iter    |  target   |  dropout  | learni... | neuronPct | neuron... |
-------------------------------------------------------------------------
| 1         | -11.59808 | 0.2080939 | 0.0720324 | 0.0101132 | 0.3093092 |
| 2         | -8.290040 | 0.0732311 | 0.0092338 | 0.1943976 | 0.3521051 |
| 3         | -11.59808 | 0.1979869 | 0.0538816 | 0.4250025 | 0.6883673 |
| 4         | -8.290040 | 0.1020216 | 0.08781